# Qa 03 static fetch routes

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA 03: Static Fetch Routes

Includes a **Local K-source curtain routing QA** section for checking that each NORAC site routes toward its own local NORA3 source cluster rather than a single global curtain.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks.multisource_notebook_helpers import (
    load_route_curtain_qa,
    load_routing_payload,
    load_sites_config,
)

try:
    import contextily as ctx
except Exception:
    ctx = None

try:
    from pyproj import Transformer
except Exception:
    Transformer = None

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
ROUTE_QA_CSV = Path("../data/processed/route_curtain_qa.csv")
ROUTING_PAYLOAD = Path("../data/processed/routing_features.pkl")
SITES_PATH = Path("../configs/sites.yaml")
COLOR_BY = "route_tortuosity_ratio"  # route_tortuosity_ratio | path_bottleneck_m | source0_distance_m | route_valid
PLOT_ALL = False
MAX_SITES = 60
RANDOM_SEED = 43
COMPARE_GLOBAL_QA = None  # optional Path('../data/processed/route_curtain_qa_global.csv')

In [ ]:
qa_df = load_route_curtain_qa(ROUTE_QA_CSV)
routing_payload = load_routing_payload(ROUTING_PAYLOAD)
routes_df = routing_payload["routes"].copy()
curtain_payload = routing_payload.get("route_curtains", []) or []
sites_cfg = load_sites_config(SITES_PATH)
nearshore_df = pd.DataFrame(sites_cfg.get("nearshore_sites", []) or [])
offshore_df = pd.DataFrame(
    [
        s
        for s in (sites_cfg.get("offshore_sites", []) or [])
        if "wind" not in str(s.get("name", "")).lower()
    ]
)
target_epsg = int((routing_payload.get("metadata", {}) or {}).get("target_epsg", 32633))
if (
    Transformer is not None
    and not offshore_df.empty
    and {"lon", "lat"}.issubset(offshore_df.columns)
):
    transformer = Transformer.from_crs(4326, target_epsg, always_xy=True)
    offshore_x, offshore_y = transformer.transform(
        offshore_df["lon"].astype(float).to_numpy(),
        offshore_df["lat"].astype(float).to_numpy(),
    )
    offshore_df = offshore_df.copy()
    offshore_df["x"] = np.asarray(offshore_x, dtype=float)
    offshore_df["y"] = np.asarray(offshore_y, dtype=float)
print("Route mode:", routing_payload.get("metadata", {}).get("route_mode"))
print("Target EPSG:", target_epsg)
print("QA rows:", len(qa_df))
print(
    "Valid routes:",
    int(pd.to_numeric(qa_df["route_valid"], errors="coerce").fillna(0).astype(int).sum()),
)
qa_df.head()

In [ ]:
# Local K-source curtain routing QA
rng = random.Random(RANDOM_SEED)
selected_sites = qa_df["site_name"].tolist()
if not PLOT_ALL and len(selected_sites) > MAX_SITES:
    selected_sites = sorted(rng.sample(selected_sites, MAX_SITES))

selected_routes = routes_df[routes_df["site_name"].isin(selected_sites)].copy()
selected_qa = qa_df[qa_df["site_name"].isin(selected_sites)].copy()
curtain_lookup = {item.get("site_name"): item for item in curtain_payload if item.get("site_name")}
offshore_lookup = (
    offshore_df.set_index("name")[["x", "y"]].to_dict("index")
    if {"name", "x", "y"}.issubset(offshore_df.columns)
    else {}
)
selected_source_rows = []
for site_name in selected_sites:
    item = curtain_lookup.get(site_name)
    if not item:
        continue
    for source in item.get("selected_sources", []) or []:
        source_name = str(source.get("name", "")).strip()
        if not source_name:
            continue
        xy = offshore_lookup.get(source_name)
        if xy is None:
            continue
        selected_source_rows.append(
            {
                "site_name": site_name,
                "source_name": source_name,
                "x": float(xy["x"]),
                "y": float(xy["y"]),
            }
        )
selected_sources_df = pd.DataFrame(selected_source_rows)
unique_selected_sources_df = (
    selected_sources_df.drop_duplicates(subset=["source_name"]).copy()
    if not selected_sources_df.empty
    else pd.DataFrame(columns=["source_name", "x", "y"])
)

fig, ax = plt.subplots(figsize=(12, 10))
values = pd.to_numeric(
    selected_qa.get(COLOR_BY, pd.Series(index=selected_qa.index, data=np.nan)), errors="coerce"
)
color_values = (
    values.to_numpy(dtype=float)
    if COLOR_BY != "route_valid"
    else selected_qa["route_valid"].astype(int).to_numpy()
)
if COLOR_BY == "route_valid":
    cmap = plt.cm.Set1
else:
    cmap = plt.cm.viridis

for _, row in selected_routes.iterrows():
    path_xy = (
        np.asarray(row["path_xy"], dtype=float)
        if isinstance(row["path_xy"], list) and len(row["path_xy"]) >= 2
        else None
    )
    if path_xy is not None and path_xy.ndim == 2 and path_xy.shape[1] == 2:
        value = selected_qa.loc[selected_qa["site_name"] == row["site_name"], COLOR_BY]
        value = pd.to_numeric(value, errors="coerce").iloc[0] if len(value) else np.nan
        color = cmap(
            0.2
            if not np.isfinite(value)
            else 0.8
            if COLOR_BY == "route_valid" and bool(value)
            else 0.1
            if COLOR_BY == "route_valid"
            else 0.5
        )
        ax.plot(path_xy[:, 0], path_xy[:, 1], color=color, linewidth=1.2, alpha=0.9)

for site_name in selected_sites:
    item = curtain_lookup.get(site_name)
    if not item:
        continue
    curtain_xy = np.asarray(item.get("curtain_xy", []), dtype=float)
    if curtain_xy.ndim == 2 and curtain_xy.shape[0] >= 2:
        ax.plot(curtain_xy[:, 0], curtain_xy[:, 1], color="black", linewidth=2.0, alpha=0.7)

# ax.scatter(offshore_df.get('x', pd.Series(dtype=float)), offshore_df.get('y', pd.Series(dtype=float)), s=26, c='#1f77b4', alpha=0.35, label='All NORA3 wave sources', zorder=4)
if not unique_selected_sources_df.empty:
    ax.scatter(
        unique_selected_sources_df["x"],
        unique_selected_sources_df["y"],
        s=90,
        c="#ff9f1c",
        edgecolor="black",
        linewidth=0.7,
        label="NORA3 points used by sampled curtains",
        zorder=6,
    )

ax.scatter(
    selected_routes["site_x"],
    selected_routes["site_y"],
    s=35,
    c="#d62728",
    label="NORAC sites",
    zorder=5,
)
ax.set_title(f"Local M-source curtain routing")
ax.set_xlabel("Projected x (m)")
ax.set_ylabel("Projected y (m)")
ax.legend(loc="best", labelcolor="white")
if ctx is not None:
    try:
        ctx.add_basemap(
            ax, crs="EPSG:32633", source=ctx.providers.Esri.WorldImagery, attribution=False, zoom=9
        )
    except Exception as exc:
        print("Basemap skipped:", exc)
ax.grid(False)
plt.show()

In [ ]:
if COMPARE_GLOBAL_QA and Path(COMPARE_GLOBAL_QA).exists():
    global_df = pd.read_csv(COMPARE_GLOBAL_QA)
    display_cols = ["site_name", "route_length_m", "route_tortuosity_ratio", "path_bottleneck_m"]
    comparison = global_df[display_cols].merge(
        qa_df[display_cols], on="site_name", suffixes=("_global", "_local")
    )
    comparison.head()
else:
    print(
        "Set COMPARE_GLOBAL_QA to compare old global-curtain QA against the new local-curtain QA."
    )